# Structural Connectivity from Diffusion MRI

**Author**: Joan Amos, Michèle Masson-Trottier

<div style="line-height: 2;">
<a href="https://github.com/joanamos"><img src="https://img.shields.io/badge/-Joan_Amos-181717?logo=github" alt="GitHub"></a><br>
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 30/03/2026

**License:**
<div style="margin-top: 10px;">
    <a href="https://opensource.org/licenses/MIT" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> MIT License
    </a>
</div>

**Note:** If this notebook uses neuroimaging tools from Neurocontainers, those tools retain their original licenses. Please see <a href="https://neurodesk.org/overview/how-to-cite-us/" target="_blank" style="color: #0066cc;">Neurodesk citation guidelines</a> for details.

### Citation and Resources

#### Tools included in this workflow

__MRtrix3__
: Tournier, J.-D., et al. (2019). MRtrix3: A fast, flexible and open software framework for medical image processing and visualisation. *NeuroImage*, 202, 116137. https://doi.org/10.1016/j.neuroimage.2019.116137

__FSL__
: Jenkinson, M., et al. (2012). FSL. *NeuroImage*, 62(2), 782–790. https://doi.org/10.1016/j.neuroimage.2011.09.015

__FreeSurfer__
: Fischl, B. (2012). FreeSurfer. *NeuroImage*, 62(2), 774–781. https://doi.org/10.1016/j.neuroimage.2012.01.021

__AFNI__
: Cox, R. W. (1996). AFNI: software for analysis and visualization of functional magnetic resonance neuroimages. *Computers and Biomedical Research*, 29(3), 162–173. https://doi.org/10.1006/cbmr.1996.0014

#### Dataset

__Human Connectome Project__
: Van Essen, D. C., et al. (2013). The Human Connectome Project: A data acquisition perspective. *NeuroImage*, 62(4), 2222–2231. https://doi.org/10.1016/j.neuroimage.2012.02.018

#### Educational resources

- MRtrix3 connectome documentation: https://mrtrix.readthedocs.io/en/latest/quantitative_structural_connectivity/structural_connectome.html
- Andy's Brain Book - MRtrix tutorial: https://andysbrainbook.readthedocs.io/en/latest/MRtrix/MRtrix_Course/MRtrix_00_Diffusion_Overview.html

## Load software tools

Load MRtrix3, FSL, AFNI, and FreeSurfer from Neurocontainers for this structural connectivity pipeline.

In [ ]:
import module
await module.load('mrtrix3/3.0.3')
await module.load('fsl/6.0.7.19')
await module.load('afni/21.2.00')
await module.load('freesurfer/7.4.1')
await module.list()

## 1. Download HCP Demo Data

Download minimally preprocessed T1w and diffusion data from a single HCP subject (100307). This requires gdown for downloading from Google Drive.

In [ ]:
%%bash
%%capture
pip install gdown

In [ ]:
%%bash
mkdir -p ~/neurodesktop-storage/Test/100307/
cd ~/neurodesktop-storage/Test/

# Note: Provide your own HCP data in ~/neurodesktop-storage/Test/100307/
# Expected files after download:
echo "Expected file structure:"
echo "  100307/aparc+aseg.nii.gz"
echo "  100307/T1w_acpc_dc_restore_brain.nii.gz"
echo "  100307/data.nii.gz  (diffusion)"
echo "  100307/bvals"
echo "  100307/bvecs"

## 2. Convert Diffusion Data to MRtrix Format

Convert the NIfTI diffusion data to MRtrix's .mif format, which embeds the gradient table for efficient processing.

In [ ]:
%%bash
cd ~/neurodesktop-storage/Test/100307/
mrconvert data.nii.gz data.mif \
    -fslgrad bvecs bvals \
    -datatype float32 \
    -strides 0,0,0,1 \
    -force
mrinfo data.mif | head -20

## 3. Estimate Response Functions

Estimate tissue-specific response functions for white matter, grey matter, and CSF using the dhollander algorithm.

In [ ]:
%%bash
cd ~/neurodesktop-storage/Test/100307/
dwi2response dhollander data.mif wm.txt gm.txt csf.txt -force
echo "Response functions estimated:"
ls -lh wm.txt gm.txt csf.txt

## 4. Generate Brain Mask and Fibre Orientation Distributions

Create a brain mask, then estimate fibre orientation distributions (FODs) for each tissue type using multi-shell multi-tissue constrained spherical deconvolution.

In [ ]:
%%bash
cd ~/neurodesktop-storage/Test/100307/
dwi2mask data.mif mask.mif -force
dwi2fod msmt_csd data.mif \
    wm.txt wmfod.mif \
    gm.txt gmfod.mif \
    csf.txt csffod.mif \
    -mask mask.mif -force
echo "FODs created"
ls -lh wmfod.mif gmfod.mif csffod.mif

## 5. Normalise FODs and Generate Tissue Segmentation

Normalise FOD amplitudes to correct for global intensity differences, then run 5ttgen to create a five-tissue-type segmentation from the T1w image.

In [ ]:
%%bash
cd ~/neurodesktop-storage/Test/100307/
mtnormalise wmfod.mif wmfod_norm.mif \
            gmfod.mif gmfod_norm.mif \
            csffod.mif csffod_norm.mif \
            -mask mask.mif -force

5ttgen fsl T1w_acpc_dc_restore_brain.nii.gz 5tt.mif \
    -premasked -force
echo "Normalisation and 5TT segmentation done"

## 6. Coregister Anatomical to Diffusion Space

Extract mean B0, coregister T1 to diffusion space with FSL FLIRT, then convert the transform for MRtrix.

In [ ]:
%%bash
cd ~/neurodesktop-storage/Test/100307/
# Extract mean B0
dwiextract data.mif - -bzero | \
    mrmath - mean mean_b0.mif -axis 3 -force

mrconvert mean_b0.mif mean_b0.nii.gz -force
mrconvert 5tt.mif 5tt.nii.gz -force

# FSL FLIRT registration
flirt -in mean_b0.nii.gz \
    -ref T1w_acpc_dc_restore_brain.nii.gz \
    -omat T1_to_DWI.mat \
    -dof 6 \
    -cost mutualinfo \
    -searchrx -30 30 -searchry -30 30 -searchrz -30 30

echo "Registration complete"
ls -lh T1_to_DWI.mat

In [ ]:
%%bash
cd ~/neurodesktop-storage/Test/100307/
# Convert FSL transform to MRtrix format
transformconvert T1_to_DWI.mat \
    mean_b0.nii.gz \
    T1w_acpc_dc_restore_brain.nii.gz \
    flirt_import \
    T1_to_DWI_mrtrix.txt -force

# Apply transform to 5TT image
mrtransform 5tt.mif \
    -linear T1_to_DWI_mrtrix.txt \
    -inverse \
    5tt_coreg.mif -force

echo "Coregistration complete"

## 7. Tractogram Generation

Generate 10 million streamlines using iFOD2 probabilistic tractography seeded from the grey matter / white matter interface.

In [ ]:
%%bash
cd ~/neurodesktop-storage/Test/100307/
# Create GM/WM interface seed mask
5tt2gmwmi 5tt_coreg.mif gmwmi_seed.mif -force

# Generate tractogram (10M streamlines for demo)
tckgen wmfod_norm.mif tracks_10M.tck \
    -act 5tt_coreg.mif \
    -seed_gmwmi gmwmi_seed.mif \
    -select 10000000 \
    -backtrack -force
echo "Tractogram generated"

## 8. SIFT2 Streamline Weighting

Weight streamlines with SIFT2 to counterbalance reconstruction biases and produce biologically plausible connectomes.

In [ ]:
%%bash
cd ~/neurodesktop-storage/Test/100307/
tcksift2 tracks_10M.tck wmfod_norm.mif sift_weights.txt \
    -act 5tt_coreg.mif \
    -out_mu sift_mu.txt -force
echo "SIFT2 complete"
ls -lh sift_weights.txt

## 9. Connectome Construction

Convert the FreeSurfer parcellation to MRtrix node labels, then build the structural connectome matrix using the Desikan-Killiany atlas (84 regions).

In [ ]:
%%bash
cd ~/neurodesktop-storage/Test/100307/

# Convert FreeSurfer parcellation using Desikan-Killiany atlas
labelconvert \
    aparc+aseg.nii.gz \
    /opt/freesurfer-7.4.1/FreeSurferColorLUT.txt \
    /usr/share/mrtrix3/labelconvert/fs_default.txt \
    nodes.mif -force

# Apply coregistration transform to parcellation
mrtransform nodes.mif \
    -linear T1_to_DWI_mrtrix.txt \
    -inverse \
    -interp nearest \
    nodes_coreg.mif -force

# Build connectome
tck2connectome tracks_10M.tck nodes_coreg.mif nodes.csv \
    -tck_weights_in sift_weights.txt \
    -symmetric -zero_diagonal -force
echo "Connectome matrix saved to nodes.csv"

## 10. Visualise the Connectome

Load and visualise the structural connectome matrix using matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

connectome_path = os.path.expanduser('~/neurodesktop-storage/Test/100307/nodes.csv')

if os.path.exists(connectome_path):
    conn = np.loadtxt(connectome_path, delimiter=',')
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(np.log1p(conn), cmap='hot', aspect='auto')
    plt.colorbar(im, ax=ax, label='log(streamline count + 1)')
    ax.set_title('Structural Connectome (Desikan-Killiany, 84 regions)', fontsize=13)
    ax.set_xlabel('Node index')
    ax.set_ylabel('Node index')
    plt.tight_layout()
    plt.show()
    print(f"Connectome shape: {conn.shape}")
    print(f"Non-zero connections: {np.count_nonzero(conn)}")
else:
    print(f"Connectome file not found at {connectome_path}")
    print("Complete the pipeline steps above first.")

## Dependencies in Jupyter/Python

Using the package [watermark](https://github.com/rasbt/watermark) to document system environment and software versions used in this notebook.

In [ ]:
%load_ext watermark
%watermark
%watermark --iversions